# Evaluation

## 1. imports 

In [5]:
import pandas as pd

## 2. Load data

In [6]:
import json
import glob
from pathlib import Path
import pandas as pd

# ── Load the most recent JSON file from the articles folder ───────
ARTICLES_DIR = "../data/articles"

json_files = sorted(glob.glob(f"{ARTICLES_DIR}/*.json"), key=lambda p: Path(p).stat().st_mtime)
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {ARTICLES_DIR}")

json_path = json_files[-1]  # most recently created/modified file
print(f"Loading: {json_path}")

with open(json_path) as f:
    articles = json.load(f)

# Flatten metadata + headline + body into rows
rows = []
for art in articles:
    row = dict(art["metadata"])
    row["headline"] = art["headline"]
    row["body"]     = art["body"]
    rows.append(row)

df = pd.DataFrame(rows)

# gold and silver flag columns
df["gold"]   = df["commodity"].isin(["gold", "gold and silver"]).astype(int)
df["silver"] = df["commodity"].isin(["silver", "gold and silver"]).astype(int)

# reorder: metadata cols, then gold/silver flags, then text
meta_cols = ["obs_id", "article_date", "series", "commodity", "gold", "silver",
             "current_price_gold", "prices_21d_gold",
             "current_price_silver", "prices_21d_silver",
             "break_in_window", "references_break",
             "n_words_target", "model"]
text_cols = ["headline", "body"]
df = df[meta_cols + text_cols]

out_path = str(Path(json_path).with_suffix(".csv"))
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")
print(f"\nGold+Silver: {((df['gold']==1) & (df['silver']==1)).sum()}")
print(f"Gold only:   {((df['gold']==1) & (df['silver']==0)).sum()}")
print(f"Silver only: {((df['gold']==0) & (df['silver']==1)).sum()}")

Loading: ../data/articles/template_gold_silver_struct_low_seed7_final_20260706_172637.json
Saved: ../data/articles/template_gold_silver_struct_low_seed7_final_20260706_172637.csv
Shape: (225, 16)

Gold+Silver: 75
Gold only:   75
Silver only: 75


## 3. Eval XGBoost

### 3.1 template_gold_silver_struct_low_seed7_final

In [7]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# Combine headline + body as input text
df["text"] = df["headline"].fillna("") + " " + df["body"].fillna("")

# TF-IDF features
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["text"])
y = df[["gold", "silver"]].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Multi-label XGBoost
clf = MultiOutputClassifier(XGBClassifier(
    n_estimators   = 200,
    max_depth      = 6,
    learning_rate  = 0.1,
    use_label_encoder = False,
    eval_metric    = "logloss",
    random_state   = 42,
    tree_method    = "hist",
))
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Results
print("=== GOLD ===")
print(classification_report(y_test[:, 0], y_pred[:, 0]))
print("=== SILVER ===")
print(classification_report(y_test[:, 1], y_pred[:, 1]))
print(f"Exact match (both correct): {np.all(y_test == y_pred, axis=1).mean():.3f}")

/home/michaelschlee/ownCloud/GIT/envs/labelFusion/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [17:30:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/michaelschlee/ownCloud/GIT/envs/labelFusion/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [17:31:00] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== GOLD ===
              precision    recall  f1-score   support

           0       0.94      0.89      0.92        19
           1       0.93      0.96      0.94        26

    accuracy                           0.93        45
   macro avg       0.94      0.93      0.93        45
weighted avg       0.93      0.93      0.93        45

=== SILVER ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        17
           1       1.00      0.96      0.98        28

    accuracy                           0.98        45
   macro avg       0.97      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45

Exact match (both correct): 0.911


In [8]:
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier

# Remove commodity keywords from text
def remove_commodity_words(text):
    text = re.sub(r'\bgold\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bsilver\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bprecious metal[s]?\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bXAU\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bXAG\b', '', text, flags=re.IGNORECASE)
    return text

df["text_clean"] = (df["headline"].fillna("") + " " + df["body"].fillna("")).apply(remove_commodity_words)

# TF-IDF features
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["text_clean"])
y = df[["gold", "silver"]].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Multi-label XGBoost
clf = MultiOutputClassifier(XGBClassifier(
    n_estimators  = 200,
    max_depth     = 6,
    learning_rate = 0.1,
    eval_metric   = "logloss",
    random_state  = 42,
    tree_method   = "hist",
))
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("=== GOLD ===")
print(classification_report(y_test[:, 0], y_pred[:, 0]))
print("=== SILVER ===")
print(classification_report(y_test[:, 1], y_pred[:, 1]))
print(f"Exact match (both correct): {np.all(y_test == y_pred, axis=1).mean():.3f}")

=== GOLD ===
              precision    recall  f1-score   support

           0       0.94      0.89      0.92        19
           1       0.93      0.96      0.94        26

    accuracy                           0.93        45
   macro avg       0.94      0.93      0.93        45
weighted avg       0.93      0.93      0.93        45

=== SILVER ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        17
           1       1.00      0.96      0.98        28

    accuracy                           0.98        45
   macro avg       0.97      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45

Exact match (both correct): 0.911
